# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryem-ahmed/flyrank-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
**Freestyle — Content Strategy Signal Analysis.** I'm going freestyle instead of the four named lanes, but I'm keeping it close to Lane 1's shape (EDA, correlations, no unearned causal claims). My question: *which `content_type` / `main_intent` / keyword-competition patterns are associated with better search visibility and engagement in the starter data, and what does that suggest about which content format FlyRank's editors should lean into next?*

Why this one: the starter data has a column I didn't expect — `provider_used` / `model_used` — meaning every page here was AI-generated. That's the part of FlyRank's business closest to my own work (I've built RAG pipelines and multi-agent systems), so a question about *which generated-content pattern actually performs* felt like the most honest place to start, rather than picking a lane because it was pre-approved. I'm treating `provider_used`/`model_used` as background context only (the data dictionary explicitly flags them as "not a model feature") — my real signal is `content_type` and `main_intent`, which are safe, non-leaky descriptive fields.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/maryem-ahmed/flyrank-internship-starter"
REPO_DIR = "flyrank-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")


assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# quick grounding check before committing to this lane: does content_type actually vary enough to say anything?
print(df["content_type"].value_counts())



Starter data found. You're ready.
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
**Decision:** which `content_type` / `main_intent` mix a content strategist should keep commissioning, throttle, or reconsider for the next batch of pages.

**Who acts:** a FlyRank content strategist or editor planning the next round of AI-generated articles for a client — this is a production-planning decision, made before new content is written, not a per-page fix.

**Cost of a wrong call:** the starter slice is overwhelmingly `keyword article` (27,207 of 30,000 pages, ~90.7%) — that's where almost all the writing budget already goes. If that format quietly underperforms on the metrics that matter (position, CTR) relative to a much smaller format the team barely uses, the cost is wasted writer/AI-generation hours and client spend on a format that isn't earning attention, while a higher-converting format stays under-scaled. Conversely, over-reacting to one AI-generated slice and shifting strategy company-wide without checking for confounds (topic, client mix, competition) risks disrupting a working pipeline over a spurious pattern. A wrong call here isn't catastrophic on any single page, but it compounds across thousands of pages and multiple clients.

**Why data/ML helps (and why not just a rule):** a simple if-statement ("always write keyword articles") is exactly the current default — the data suggests format-level differences (see Section 3) that a flat rule can't see, because they only show up when you group and compare, not from any single page. That said, this stays at the EDA/signal-analysis level (grouped comparisons, effect sizes) rather than a trained model — with only 3 content_type values and a handful of intents, this is squarely "which signals travel together," not a ranking or classification task yet.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantifying the scale of the decision: how much of the pipeline's reach sits in each content_type?
scale = df.groupby("content_type").agg(
    n_pages=("content_id", "count"),
    total_impressions_90d=("impressions_90d", "sum"),
)
scale["share_of_pages_pct"] = (scale["n_pages"] / len(df) * 100).round(1)
scale["share_of_impressions_pct"] = (scale["total_impressions_90d"] / df["impressions_90d"].sum() * 100).round(1)
scale

,n_pages,total_impressions_90d,share_of_pages_pct,share_of_impressions_pct
content_type,,,,
comparison article,697,183679,2.3,0.1
feedly article,2096,395370,7.0,0.3
keyword article,27207,155431940,90.7,99.6


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
Three numbers from the code cell below, in plain words:

1. **90.7% of pages (27,207 / 30,000) are `keyword article`**, but they average CTR ≈ 0.34% — the *lowest* of the three content types — while carrying 155.4M of the dataset's ~156.0M total 90-day impressions (99.6% of all reach).
2. **`feedly article` pages average CTR ≈ 2.79%** (about 8x the keyword-article average) and a better average search position (≈8.9 vs ≈17.6, excluding `avg_position = 0` "no data" rows) — but they only make up 7.0% of pages and a much smaller slice of total impressions (395K, 0.25%), and their median impressions-per-page is just 4.
3. **`main_intent` is 100% missing for every `feedly article` row (2,096/2,096)**, and `comparison article` is 100% `informational` intent (697/697) — so intent and content_type are entangled in this slice, not independent signals. That's a real gotcha (matches the "missingness follows content_type" warning in the data skill), not something I can casually fillna() around.

Worth 7 weeks: there's a real, sizeable gap between the format that gets almost all the writing budget (`keyword article`) and the format that converts best per impression (`feedly article`) — but the volume/reach story cuts the other way, and intent data is missing exactly where I'd want it most. That tension — "which one should scale?" — is a real production-planning question with a genuinely messy signal, not something one glance at a dashboard answers.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Real numbers backing Section 3

# 1) volume vs performance split by content_type
by_type = df.groupby("content_type").agg(
    n_pages=("content_id", "count"),
    avg_ctr_pct=("ctr", "mean"),
    avg_position=("avg_position", lambda s: s[s > 0].mean()),  # avg_position==0 means "no data" (data dictionary)
    total_impressions_90d=("impressions_90d", "sum"),
    median_impressions_per_page=("impressions_90d", "median"),
).round(2)
print("=== Performance by content_type ===")
print(by_type)
print()

# 2) intent x content_type — how entangled are they?
print("=== main_intent missingness / concentration by content_type ===")
for ct in df["content_type"].unique():
    sub = df.loc[df["content_type"] == ct, "main_intent"]
    print(f"{ct:20s} n={len(sub):5d}  missing_intent={sub.isna().sum():5d}  top_intent={sub.mode().iloc[0] if sub.notna().any() else 'NONE'}")


=== Performance by content_type ===
                    n_pages  avg_ctr_pct  avg_position  total_impressions_90d  \
content_type                                                                    
comparison article      697         0.13         11.30                 183679   
feedly article         2096         2.79          8.86                 395370   
keyword article       27207         0.34         17.59              155431940   

                    median_impressions_per_page  
content_type                                     
comparison article                        107.0  
feedly article                              4.0  
keyword article                           955.0  

=== main_intent missingness / concentration by content_type ===
keyword article      n=27207  missing_intent=  278  top_intent=informational
feedly article       n= 2096  missing_intent= 2096  top_intent=NONE
comparison article   n=  697  missing_intent=    0  top_intent=informational


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
**What I can say (observed):** in this 30,000-row, 32-client slice, `feedly article` pages have a higher average CTR and a better average search position than `keyword article` pages, while carrying far fewer impressions per page and almost none of the total reach. These are measured group differences in the trailing-90-day window, not a claim about why.

**What I can say (directional / decision-support):** the gap is large enough, and the volume-vs-performance tradeoff is stark enough, that it's *worth* a content strategist reviewing whether `feedly article` deserves more production budget, and whether `keyword article`'s huge reach is masking weak per-page performance. This is a candidate for a human review, not an automatic switch.

**What I will never claim:**
- That `content_type` *causes* the CTR/position difference — content_type, main_intent, client mix, topic, and keyword competition are all tangled together here (main_intent is literally missing for every feedly row), and I have no experiment, only observation.
- That I've "predicted Google's algorithm" or reverse-engineered a ranking factor — `avg_position` and `ctr` are outcomes I'm describing, not a model of search behavior.
- That this generalizes beyond these 32 pseudonymized clients and this 90-day window — different clients, topics, or seasons could easily shift these numbers.
- That `provider_used` / `model_used` (which LLM generated the article) is safe to treat as a real signal — the data dictionary flags both as "not a model feature," and I haven't touched them beyond noting AI-generation is the norm here, not the exception.

**Leakage check (from the data skill):** I did not use `trend_direction` or `trend_pct` anywhere above — those define the pre-built `is_declining_label` and are never legitimate inputs, even for descriptive stats, since they'd just restate a rule someone already wrote.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Guard-rail check: confirm I never touched the leakage-flagged columns above
leaky_cols = {"trend_direction", "trend_pct", "is_declining_label"}
used_in_this_notebook = {"content_type", "main_intent", "ctr", "avg_position", "impressions_90d", "content_id", "client_id"}
assert leaky_cols.isdisjoint(used_in_this_notebook), "leakage-flagged column used in analysis!"
print("No leakage-flagged columns (trend_direction / trend_pct / is_declining_label) used. Clear to proceed.")


No leakage-flagged columns (trend_direction / trend_pct / is_declining_label) used. Clear to proceed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.